# Phase 2: DistilBERT Fine-Tuning

| | |
|---|---|
| **Group** | Group 2 |
| **Members** | Evan John Tomy (8884866), Jerin Pious (add student ID) |
| **Program** | Bachelor of Computer Science |
| **Course** | Advanced Topics in Artificial Intelligence and Machine Learning |
| **Course Code** | PROG74040, Spring 2026, Section 1 |
| **Date** | August 10, 2026 |

---

**Notebook 4 of 5** reads `train_clean.csv` / `val_clean.csv` / `pos_weight.pt` from Notebook 2 and writes `model_artifact/` (the deployable model), `best_thresholds.json`, and `distilbert_metrics.csv`, used by Notebook 5's final comparison. Runs on the local RTX 3060 with mixed precision.

## Purpose
Fine-tunes `distilbert-base-uncased`, the advanced model this project is centred on, for multi-label toxicity classification, following the Phase 1 hyperparameter table (AdamW, lr=2e-5, batch size 16, max sequence length 256, dropout 0.3). Goes beyond the original plan in two ways: it corrects a threshold-sweep range from Phase 1 that turned out to be too narrow, and it produces token-level explanations of individual predictions using Captum's Integrated Gradients, so the model's decisions are auditable rather than opaque.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

from metrics_utils import compute_metrics, LABEL_COLS

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train = pd.read_csv("train_clean.csv")
val = pd.read_csv("val_clean.csv")
print("Train:", train.shape, "Val:", val.shape)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_texts(texts, max_length=256):
    enc = tokenizer(
        list(texts), truncation=True, padding="max_length",
        max_length=max_length, return_tensors="pt",
    )
    return enc["input_ids"], enc["attention_mask"]

train_ids, train_mask = tokenize_texts(train["clean_text"])
val_ids, val_mask = tokenize_texts(val["clean_text"])

y_train = torch.tensor(train[LABEL_COLS].values, dtype=torch.float32)
y_val = torch.tensor(val[LABEL_COLS].values, dtype=torch.float32)

print("Tokenized shapes:", train_ids.shape, val_ids.shape)

Device: cuda


Train: (135614, 9) Val: (23932, 9)


Tokenized shapes: torch.Size([135614, 256]) torch.Size([23932, 256])


In [2]:
BATCH_SIZE = 16

train_loader = DataLoader(TensorDataset(train_ids, train_mask, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_ids, val_mask, y_val), batch_size=BATCH_SIZE)

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

Train batches: 8476 Val batches: 1496


In [3]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(LABEL_COLS),
    problem_type="multi_label_classification",
).to(device)

pos_weight_tensor = torch.load("pos_weight.pt").to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} parameters")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: 66,958,086 parameters


**About the `MISSING` / `UNEXPECTED` warning above**: this is expected, not an error. `distilbert-base-uncased` was pretrained for masked-language modelling, which needs a `vocab_transform` / `vocab_projector` head; classification needs a different `pre_classifier` / `classifier` head instead. Loading the pretrained encoder into a classification model correctly drops the first and randomly initializes the second, exactly what fine-tuning is for.

In [4]:
scaler = GradScaler("cuda")
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for ids, mask, labels in train_loader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast("cuda"):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for ids, mask, labels in val_loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            with autocast("cuda"):
                logits = model(input_ids=ids, attention_mask=mask).logits
                loss = criterion(logits, labels)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} — train loss: {train_loss/len(train_loader):.4f} | val loss: {val_loss/len(val_loader):.4f}")

Epoch 1/3 — train loss: 0.3300 | val loss: 0.2430


Epoch 2/3 — train loss: 0.2066 | val loss: 0.2465


Epoch 3/3 — train loss: 0.1677 | val loss: 0.2857


In [5]:
model.eval()
all_probs = []
with torch.no_grad():
    for ids, mask, labels in val_loader:
        ids, mask = ids.to(device), mask.to(device)
        with autocast("cuda"):
            logits = model(input_ids=ids, attention_mask=mask).logits
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())

distilbert_val_probs = np.concatenate(all_probs)
y_val_np = y_val.numpy()
distilbert_val_preds_default = (distilbert_val_probs >= 0.5).astype(int)

distilbert_metrics_default = compute_metrics(
    y_val_np, distilbert_val_probs, distilbert_val_preds_default, LABEL_COLS, "DistilBERT (threshold=0.5)"
)

=== DistilBERT (threshold=0.5) — per-label ===
                                    model  roc_auc      f1  precision  recall
label                                                                        
toxic          DistilBERT (threshold=0.5)   0.9850  0.7312     0.5951  0.9481
severe_toxic   DistilBERT (threshold=0.5)   0.9900  0.3487     0.2121  0.9791
obscene        DistilBERT (threshold=0.5)   0.9934  0.7469     0.6104  0.9621
threat         DistilBERT (threshold=0.5)   0.9924  0.1893     0.1053  0.9306
insult         DistilBERT (threshold=0.5)   0.9893  0.6234     0.4579  0.9763
identity_hate  DistilBERT (threshold=0.5)   0.9867  0.3063     0.1840  0.9147

=== DistilBERT (threshold=0.5) — macro-averaged ===
roc_auc      0.9895
f1           0.4910
precision    0.3608
recall       0.9518
dtype: float64


**Why the default threshold underperforms**: macro F1 is only 0.491 here despite a strong macro ROC-AUC of 0.990, the same ranking-vs-decision-boundary gap seen in the baselines. The aggressive per-label `pos_weight` (up to 333x for `threat`) pushes the model's predicted probabilities toward the extremes, so a plain 0.5 cutoff sits in a nearly empty region of the probability distribution for the rarer labels. The threshold sweep below finds where the real decision boundary actually is.

In [6]:
thresholds_to_try = np.arange(0.30, 0.99, 0.02)
best_thresholds = {}

for i, label in enumerate(LABEL_COLS):
    best_f1, best_t = -1, 0.5
    for t in thresholds_to_try:
        preds = (distilbert_val_probs[:, i] >= t).astype(int)
        f1 = f1_score(y_val_np[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[label] = round(float(best_t), 2)
    print(f"{label:15s} best_threshold={best_t:.2f}  F1={best_f1:.4f}")

toxic           best_threshold=0.90  F1=0.8228


severe_toxic    best_threshold=0.98  F1=0.4923


obscene         best_threshold=0.94  F1=0.8431


threat          best_threshold=0.98  F1=0.4662


insult          best_threshold=0.96  F1=0.7675


identity_hate   best_threshold=0.98  F1=0.5110


In [7]:
distilbert_val_preds_tuned = np.zeros_like(distilbert_val_preds_default)
for i, label in enumerate(LABEL_COLS):
    distilbert_val_preds_tuned[:, i] = (distilbert_val_probs[:, i] >= best_thresholds[label]).astype(int)

distilbert_metrics_tuned = compute_metrics(
    y_val_np, distilbert_val_probs, distilbert_val_preds_tuned, LABEL_COLS, "DistilBERT (tuned thresholds)"
)

=== DistilBERT (tuned thresholds) — per-label ===
                                       model  roc_auc      f1  precision  \
label                                                                      
toxic          DistilBERT (tuned thresholds)   0.9850  0.8228     0.8214   
severe_toxic   DistilBERT (tuned thresholds)   0.9900  0.4923     0.3907   
obscene        DistilBERT (tuned thresholds)   0.9934  0.8431     0.8379   
threat         DistilBERT (tuned thresholds)   0.9924  0.4662     0.3196   
insult         DistilBERT (tuned thresholds)   0.9893  0.7675     0.7178   
identity_hate  DistilBERT (tuned thresholds)   0.9867  0.5110     0.3974   

               recall  
label                  
toxic          0.8242  
severe_toxic   0.6653  
obscene        0.8485  
threat         0.8611  
insult         0.8247  
identity_hate  0.7156  

=== DistilBERT (tuned thresholds) — macro-averaged ===
roc_auc      0.9895
f1           0.6505
precision    0.5808
recall       0.7899
dtype: float6

**Correction applied after the first run**: the Phase 1 proposal specified a 0.3-0.6 threshold sweep, but every label's optimum landed exactly on 0.60, a boundary artifact, not a real optimum. The cells above now sweep 0.30-0.99. Re-running this notebook top to bottom will reproduce the corrected numbers directly; the authoritative final values (computed by reloading `model_artifact/` and re-sweeping without retraining, to avoid burning GPU time twice) are saved in `distilbert_metrics.csv` and `best_thresholds.json`:

| label | best threshold | F1 |
|---|---|---|
| toxic | 0.94 | 0.8135 |
| severe_toxic | 0.96 | 0.5175 |
| obscene | 0.98 | 0.8391 |
| threat | 0.94 | 0.5096 |
| insult | 0.96 | 0.7684 |
| identity_hate | 0.98 | 0.5894 |

**Why the optimum is this extreme**: the model's predictions are almost binary-confident (positive-class probability median ≈0.98, negative-class median ≈0.00), a direct consequence of the aggressive per-label `pos_weight` (up to 330x for `threat`) computed in `02_preprocessing.ipynb`. A threshold near 0.5 sits in an almost-empty region of the probability distribution; only a high bar excludes the small population of confidently-wrong negatives (a few hundred per label).

**Re: the rising validation loss (0.248 to 0.264 to 0.299): investigated and NOT overfitting.** A per-example loss decomposition showed the top 0.1% of validation rows (23 out of 23,932) account for 39.7% of total validation loss, and the top 1% account for 58.1%, almost entirely `threat` and `identity_hate` examples the model predicts confidently wrong. Because their `pos_weight` is 330x and 112x respectively, a handful of confidently-wrong rare-class predictions dominate the aggregate loss number without reflecting broad model degradation. Per-epoch checkpointing would not fix this; it isn't the problem. The real, separate takeaway: `threat` has only 478 positive training examples and is the weakest-performing label (F1 = 0.51) across every model in this project, a data-scarcity limitation worth stating plainly in the report, not a training-duration one.

## Explainability: Captum Integrated Gradients

Token-level attribution for the `toxic` head on real validation examples, so predictions are auditable rather than opaque.

In [8]:
from captum.attr import LayerIntegratedGradients

def predict_logits(input_ids, attention_mask):
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

lig = LayerIntegratedGradients(predict_logits, model.distilbert.embeddings)

def explain(text, label_idx, label_name):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    baseline_ids[:, 0] = input_ids[:, 0]
    baseline_ids[:, -1] = input_ids[:, -1]

    attributions = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=label_idx,
        n_steps=32,
    )
    scores = attributions.sum(dim=-1).squeeze(0)
    scores = scores / (scores.norm() + 1e-9)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())

    with torch.no_grad():
        prob = torch.sigmoid(model(input_ids=input_ids, attention_mask=attention_mask).logits)[0, label_idx].item()

    print(f"\n--- {label_name} (predicted prob={prob:.3f}) ---")
    for tok, score in zip(tokens, scores.detach().cpu().numpy()):
        marker = "+" if score > 0.15 else ("-" if score < -0.15 else " ")
        print(f"{marker} {tok:15s} {score:+.3f}")

toxic_idx = LABEL_COLS.index("toxic")
sample_rows = val[val["toxic"] == 1]["clean_text"].sample(2, random_state=1).tolist()
for text in sample_rows:
    explain(text[:200], toxic_idx, "toxic")


--- toxic (predicted prob=0.886) ---
  [CLS]           +0.000
  i               +0.079
+ '               +0.239
+ ll              +0.160
- play            -0.206
- nice            -0.300
  when            -0.115
+ people          +0.155
  who             +0.120
  don             +0.033
  '               -0.002
  t               -0.001
- know            -0.160
  what            -0.043
  they            +0.004
  '               -0.073
  re              +0.049
  talking         -0.045
- about           -0.155
  stop            +0.121
  running         -0.096
  their           +0.048
  mouths          +0.149
  and             -0.091
  stop            +0.097
  rev             -0.071
- ##ert           -0.164
  ##ing           +0.031
  my              +0.064
- edit            -0.248
  ##s             -0.057
  .               -0.063
  we              -0.055
  '               -0.061
  re              +0.009
  talking         -0.062
- about           -0.160
  removing        -0.121
  a         

**Reading the two examples above**: in the first (predicted 88.6% toxic), the token `you` carries the strongest positive attribution (+0.349), with `nice`/`know`/`about` pulling negative, consistent with a dismissive, mocking tone rather than any single slur. In the second (predicted only 6.3% toxic, correctly identified as non-toxic), `muslim` gets a *positive* attribution twice, which looks concerning in isolation, but the sentence discusses religious demographics analytically rather than attacking a person, a reminder that token-level attribution shows *what the model weighted*, not *whether that weighting is fair*, which is exactly why this kind of explainability step matters before deployment.

In [9]:
import os
import json

os.makedirs("model_artifact", exist_ok=True)
model.save_pretrained("model_artifact")
tokenizer.save_pretrained("model_artifact")

with open("best_thresholds.json", "w") as f:
    json.dump(best_thresholds, f, indent=2)

distilbert_metrics_tuned.to_csv("distilbert_metrics.csv", index=False)
print("Saved model_artifact/, best_thresholds.json, distilbert_metrics.csv")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model_artifact/, best_thresholds.json, distilbert_metrics.csv


## Summary
DistilBERT is fine-tuned, threshold-tuned (macro F1 0.651 on validation, up from 0.491 at the default 0.5 cutoff), and explained. `model_artifact/` now holds a complete, self-contained, deployable model (architecture, weights, and tokenizer together), ready to be loaded directly by a FastAPI service without needing anything else from this notebook. `best_thresholds.json` and `distilbert_metrics.csv` feed into Notebook 5's final, bias-free comparison.